# HDD 22 - Nuisance-controlled regime ablation

Notebook 21 found that some frozen-encoder clusters mostly reflect drive age (SMART 9) or positive sector counts, while ST12000NM0008 is unstable. This controlled, **label-free diagnostic** subtracts the part of the frozen embedding predicted by five final-day SMART values and sector-positive status. It then repeats per-model clustering and temporal persistence checks on residual embeddings. The regression and clustering use only August 15 fit-training drives; stop drives and August 1/8 snapshots are held out from fitting. No failure labels, evaluation drives, Q4, or sealed June are opened.

Residualization is an exploratory probe, **not** an approved production transformation: it can remove legitimate information and leaves nonlinear associations. Interpret multiple reasonably sized, separated, stable clusters persisting beyond a shuffled-drive control as evidence for further testing, not as proof of a healthy/failing regime. If it collapses to one cluster, prioritize the established reconstruction and SMART baselines rather than force a regime component into alerts.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib,json,re,zipfile
import numpy as np
import pandas as pd
from sklearn.cluster import HDBSCAN
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import adjusted_mutual_info_score,silhouette_score
from sklearn.preprocessing import RobustScaler,StandardScaler
try:
    from IPython.display import display
except ImportError:
    display=print
COMPONENT_ROOT=Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID='single_model_q1q2_2021_v1'
REGIME_ID='q3_frozen_encoder_regime_feasibility_v1'
AUDIT_ID='q3_regime_stability_smart_audit_v1'
ABLATION_ID='q3_nuisance_controlled_regime_ablation_v1'
REGIME=COMPONENT_ROOT/'results'/RUN_ID/REGIME_ID
AUDIT=COMPONENT_ROOT/'results'/RUN_ID/AUDIT_ID
OUT=COMPONENT_ROOT/'results'/RUN_ID/ABLATION_ID
OUT.mkdir(parents=True,exist_ok=True)
Q3_ZIP=COMPONENT_ROOT/'data/raw/data_Q3_2021.zip'
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()
def read_json(path):
    assert path.is_file(),f'Missing prerequisite: {path}'
    return json.loads(path.read_text())
r20=read_json(REGIME/'regime_feasibility_complete.json')
r21=read_json(AUDIT/'regime_audit_complete.json')
s20=read_json(REGIME/'settings.json')
assert r20['analysis_complete'] and r21['analysis_complete']
assert not r20['failure_labels_used'] and not r21['failure_labels_used']
assert not r20['evaluation_drives_used'] and not r21['evaluation_drives_used']
assert not r20['future_test_opened'] and not r21['future_test_opened']
assert sha256(Q3_ZIP)==s20['q3_zip_sha256']
embedding_path=REGIME/'three_snapshot_fit_embeddings.parquet'
assert sha256(embedding_path)==r20['embeddings_sha256']
embeddings=pd.read_parquet(embedding_path)
assert not embeddings.duplicated(['serial_number','end_date']).any()
SNAPSHOTS=s20['snapshots'];assert SNAPSHOTS==['2021-08-01','2021-08-08','2021-08-15']
models=list(read_json(REGIME/'regime_projection_spec.json'))
assert set(embeddings.model)==set(models) and set(embeddings.end_date)==set(SNAPSHOTS)
features=[x for x in embeddings if x.startswith('embed_')];assert len(features)==64
nuisance=['smart_5_raw','smart_9_raw','smart_194_raw','smart_197_raw','smart_198_raw']
SETTINGS={'notebook20_complete_sha256':sha256(REGIME/'regime_feasibility_complete.json'),
          'notebook21_complete_sha256':sha256(AUDIT/'regime_audit_complete.json'),
          'embeddings_sha256':sha256(embedding_path),'q3_zip_sha256':sha256(Q3_ZIP),
          'snapshots':SNAPSHOTS,'nuisance_features':nuisance,
          'nuisance_transform':'log1p five raw values plus binary sector-positive; StandardScaler on Aug15 train',
          'residualizer':'per-model Ridge alpha=10; predicts 64D embedding from nuisance',
          'residual_pca_dimensions':8,'cluster_min_size_fraction':.01,'cluster_min_size_floor':30,
          'min_samples':15,'projection':'nearest centroid within train-cluster 95th percentile radius',
          'shuffle_repetitions':100,'seed':42,'labels_used':False,'production_ready':False}
settings_path=OUT/'settings.json'
if settings_path.exists():assert read_json(settings_path)==SETTINGS,'Change ABLATION_ID before changing the protocol.'
else:settings_path.write_text(json.dumps(SETTINGS,indent=2))
print('Frozen embeddings and prior audit verified. Reading only three Q3 SMART snapshots.')

Frozen embeddings and prior audit verified. Reading only three Q3 SMART snapshots.


## 1. Match final-day SMART data to frozen embedding keys

Read only the three dated archive members and the five named SMART columns. Preserve the exact fit-drive serial/date/model population from Notebook 20; any mismatch stops execution. The Backblaze `failure` column is never requested.

In [2]:
with zipfile.ZipFile(Q3_ZIP) as z:
    members={}
    for info in z.infolist():
        if info.is_dir() or '__MACOSX' in info.filename:continue
        match=re.fullmatch(r'(2021-\d{2}-\d{2})\.csv',Path(info.filename).name)
        if match and match.group(1) in SNAPSHOTS:
            assert match.group(1) not in members
            members[match.group(1)]=info.filename
assert set(members)==set(SNAPSHOTS)
chunks=[]
with zipfile.ZipFile(Q3_ZIP) as z:
    for date in SNAPSHOTS:
        expected=embeddings.loc[embeddings.end_date.eq(date),['serial_number','model','end_date','subset']]
        wanted=set(expected.serial_number)
        seen=set()
        with z.open(members[date]) as f:
            for chunk in pd.read_csv(f,usecols=['date','serial_number','model']+nuisance,
                                     dtype={'serial_number':str},chunksize=50000):
                part=chunk.loc[chunk.serial_number.isin(wanted)].copy()
                if part.empty:continue
                assert part.date.astype(str).eq(date).all()
                ids=part.serial_number.tolist()
                assert len(ids)==len(set(ids)) and seen.isdisjoint(ids)
                seen.update(ids)
                chunks.append(part.rename(columns={'date':'end_date'}))
        assert seen==wanted,(date,len(wanted),len(seen))
raw=pd.concat(chunks,ignore_index=True)
assert not raw.duplicated(['serial_number','end_date']).any()
for col in nuisance:raw[col]=pd.to_numeric(raw[col],errors='coerce')
assert np.isfinite(raw[nuisance].to_numpy(dtype=float)).all() and raw[nuisance].ge(0).all().all()
joined=embeddings.merge(raw,on=['serial_number','model','end_date'],how='outer',validate='one_to_one',indicator=True)
assert len(joined)==len(embeddings)==len(raw) and joined._merge.eq('both').all()
joined.drop(columns='_merge',inplace=True)
joined['sector_positive']=joined[['smart_5_raw','smart_197_raw','smart_198_raw']].gt(0).any(axis=1)
assert set(joined.subset)=={'train','stop'}
print('Matched snapshot rows:',len(joined))

Matched snapshot rows: 138965


## 2. Remove simple SMART context, then cluster residual embeddings

For each model, fit StandardScaler and Ridge on August 15 training drives only. Apply that fixed map to the earlier snapshots and stop drives. Fit RobustScaler, PCA and HDBSCAN on August 15 training residuals. Store exact transform coefficients, centroids, radii and all assignments. HDBSCAN noise (`-1`) is preserved.

In [3]:
tables=[];configs={};summaries=[]
for model in models:
    group=joined.loc[joined.model.eq(model)].copy()
    train=group.loc[group.end_date.eq('2021-08-15')&group.subset.eq('train')].copy()
    assert len(train)>200
    def raw_nuisance(frame):
        return np.column_stack([np.log1p(frame[nuisance].to_numpy(dtype=float)),
                                frame.sector_positive.to_numpy(dtype=float)])
    scaler=StandardScaler().fit(raw_nuisance(train))
    regression=Ridge(alpha=10.0).fit(scaler.transform(raw_nuisance(train)),train[features].to_numpy(dtype=float))
    residual=group[features].to_numpy(dtype=float)-regression.predict(scaler.transform(raw_nuisance(group)))
    assert residual.shape==(len(group),len(features)) and np.isfinite(residual).all()
    residual_table=pd.DataFrame(residual,index=group.index,columns=features)
    train_residual=residual_table.loc[train.index].to_numpy(dtype=float)
    robust=RobustScaler().fit(train_residual)
    pca=PCA(n_components=8,svd_solver='full').fit(robust.transform(train_residual))
    train_pca=pca.transform(robust.transform(train_residual))
    min_size=max(SETTINGS['cluster_min_size_floor'],int(np.ceil(len(train)*SETTINGS['cluster_min_size_fraction'])))
    labels=HDBSCAN(min_cluster_size=min_size,min_samples=SETTINGS['min_samples'],
                   allow_single_cluster=False).fit_predict(train_pca).astype(int)
    cluster_ids=sorted(set(labels)-{-1});centroids={};radii={}
    for cid in cluster_ids:
        center=train_pca[labels==cid].mean(axis=0)
        distances=np.linalg.norm(train_pca[labels==cid]-center,axis=1)
        centroids[cid]=center
        radii[cid]=float(np.quantile(distances,.95,method='higher'))
    all_pca=pca.transform(robust.transform(residual_table.to_numpy(dtype=float)))
    projected=np.full(len(group),-1,dtype=int)
    distance=np.full(len(group),np.nan)
    if cluster_ids:
        centers=np.stack([centroids[cid] for cid in cluster_ids])
        ds=np.linalg.norm(all_pca[:,None,:]-centers[None,:,:],axis=2)
        nearest=ds.argmin(axis=1)
        distance=ds[np.arange(len(group)),nearest]
        projected=np.array([cluster_ids[j] if distance[i]<=radii[cluster_ids[j]] else -1 for i,j in enumerate(nearest)])
    group['residual_regime']=projected
    group['distance_to_prototype']=distance
    group['hdbscan_fit_label']=pd.Series(pd.NA,index=group.index,dtype='Int64')
    group.loc[train.index,'hdbscan_fit_label']=labels
    tables.append(group[['serial_number','model','end_date','subset','residual_regime','distance_to_prototype','hdbscan_fit_label']])
    valid=np.flatnonzero(labels!=-1)
    if len(cluster_ids)>=2 and len(valid)>len(cluster_ids):
        rng=np.random.default_rng(SETTINGS['seed'])
        chosen=rng.choice(valid,size=min(len(valid),3000),replace=False)
        silhouette=float(silhouette_score(train_pca[chosen],labels[chosen])) if len(set(labels[chosen]))>=2 else np.nan
    else:silhouette=np.nan
    shares=pd.Series(labels[labels!=-1]).value_counts(normalize=True)
    summaries.append({'model':model,'fit_train_windows':len(train),'cluster_count':len(cluster_ids),
                      'fit_noise_fraction':float((labels==-1).mean()),
                      'largest_assigned_cluster_share':float(shares.max()) if len(shares) else np.nan,
                      'silhouette_assigned_sample':silhouette,
                      'stop_projected_coverage':float((group.loc[group.end_date.eq('2021-08-15')&group.subset.eq('stop'),'residual_regime']!=-1).mean()),
                      'ridge_train_R2':float(regression.score(scaler.transform(raw_nuisance(train)),train[features].to_numpy(dtype=float)))})
    configs[model]={'nuisance_mean':scaler.mean_.tolist(),'nuisance_scale':scaler.scale_.tolist(),
                    'ridge_coef':regression.coef_.tolist(),'ridge_intercept':regression.intercept_.tolist(),
                    'residual_center':robust.center_.tolist(),'residual_scale':robust.scale_.tolist(),
                    'pca_mean':pca.mean_.tolist(),'pca_components':pca.components_.tolist(),
                    'min_cluster_size':min_size,'min_samples':SETTINGS['min_samples'],
                    'cluster_sizes':{str(cid):int((labels==cid).sum()) for cid in cluster_ids},
                    'centroids':{str(cid):centroids[cid].tolist() for cid in cluster_ids},
                    'radii_95':{str(cid):radii[cid] for cid in cluster_ids}}
assignment=pd.concat(tables,ignore_index=True)
summary=pd.DataFrame(summaries)
assert len(assignment)==len(joined) and not assignment.duplicated(['serial_number','end_date']).any()
assignment.to_csv(OUT/'residual_regime_assignments.csv',index=False)
summary.to_csv(OUT/'residual_cluster_summary.csv',index=False)
(OUT/'residual_projection_spec.json').write_text(json.dumps(configs,indent=2))
display(summary)

c:\Users\IMAKA\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\IMAKA\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\IMAKA\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\IMAKA\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly 

,model,fit_train_windows,cluster_count,fit_noise_fraction,largest_assigned_cluster_share,silhouette_assigned_sample,stop_projected_coverage,ridge_train_R2
0,TOSHIBA MG07ACA14TA,13825,2,0.048825,0.963726,0.317063,0.854745,0.851200
1,ST12000NM0008,8180,2,0.415770,0.796401,0.608774,0.795139,0.914250
2,ST4000DM000,8130,2,0.036162,0.989408,0.628017,0.923077,0.964668
3,ST8000NM0055,6173,2,0.018954,0.971598,0.904535,0.921852,0.889207
4,HGST HMS5C4040BLE640,5448,2,0.009545,0.984989,0.646590,0.948518,0.875495


## 3. Compare confounds and temporal persistence with the original clusters

Report age and sector association in the August 15 fit-training subset. For each adjacent snapshot pair, compare the same drive’s residual regime, and subtract a shuffled-drive same-regime rate that preserves cluster proportions. The old values are displayed alongside new values; the comparison is descriptive because residualization and clustering have different geometries and coverage.

In [4]:
metrics=[];temporal=[]
original=pd.read_csv(AUDIT/'regime_feature_associations.csv')
assert sha256(AUDIT/'regime_feature_associations.csv')==r21['association_sha256']
old_persistence=pd.read_csv(REGIME/'temporal_persistence_vs_shuffle.csv')
assert sha256(REGIME/'temporal_persistence_vs_shuffle.csv')==r20['persistence_sha256']
for model in models:
    group=assignment.loc[assignment.model.eq(model)]
    f=joined.loc[joined.model.eq(model)&joined.end_date.eq('2021-08-15')&joined.subset.eq('train')]
    ids=group.loc[group.end_date.eq('2021-08-15')&group.subset.eq('train'),['serial_number','hdbscan_fit_label']]
    sample=f.merge(ids,on='serial_number',validate='one_to_one')
    valid=sample.loc[sample.hdbscan_fit_label.ne(-1)]
    labels=valid.hdbscan_fit_label.to_numpy(dtype=int)
    edges=np.unique(np.quantile(f.smart_9_raw.to_numpy(dtype=float),[0,.2,.4,.6,.8,1]))
    age_bins=np.searchsorted(edges[1:-1],valid.smart_9_raw.to_numpy(dtype=float),side='right') if len(edges)>=3 else None
    metrics.append({'model':model,'residual_age_AMI':float(adjusted_mutual_info_score(labels,age_bins)) if age_bins is not None and len(set(labels))>1 else np.nan,
                    'residual_sector_AMI':float(adjusted_mutual_info_score(labels,valid.sector_positive.astype(int))) if len(set(labels))>1 else np.nan})
    for left,right in zip(SNAPSHOTS[:-1],SNAPSHOTS[1:]):
        a=group.loc[group.end_date.eq(left),['serial_number','residual_regime']].rename(columns={'residual_regime':'left_regime'})
        b=group.loc[group.end_date.eq(right),['serial_number','residual_regime']].rename(columns={'residual_regime':'right_regime'})
        pair=a.merge(b,on='serial_number',validate='one_to_one')
        assert len(pair)>100
        x=pair.left_regime.to_numpy(dtype=int);y=pair.right_regime.to_numpy(dtype=int)
        both=(x!=-1)&(y!=-1)
        observed=float((x[both]==y[both]).mean()) if both.any() else np.nan
        rng=np.random.default_rng(SETTINGS['seed']+SNAPSHOTS.index(right))
        shuffled=[]
        for _ in range(SETTINGS['shuffle_repetitions']):
            perm=rng.permutation(y);mask=(x!=-1)&(perm!=-1)
            shuffled.append(float((x[mask]==perm[mask]).mean()) if mask.any() else np.nan)
        null=float(np.nanmean(shuffled)) if np.isfinite(shuffled).any() else np.nan
        temporal.append({'model':model,'left_date':left,'right_date':right,'matched_drives':len(pair),
                         'both_assigned_fraction':float(both.mean()),'observed_same_regime':observed,
                         'shuffled_same_regime_mean':null,
                         'excess_same_regime':observed-null if np.isfinite(observed) and np.isfinite(null) else np.nan})
association=pd.DataFrame(metrics).merge(original[['model','age_quintile_AMI','sector_flag_AMI']],on='model',validate='one_to_one')
association.to_csv(OUT/'residual_vs_original_confounds.csv',index=False)
temporal=pd.DataFrame(temporal)
assert len(temporal)==len(old_persistence)
comparison=temporal.merge(old_persistence[['model','left_date','right_date','excess_same_regime','both_assigned_fraction']],
    on=['model','left_date','right_date'],validate='one_to_one',suffixes=('_residual','_original'))
comparison.to_csv(OUT/'residual_vs_original_persistence.csv',index=False)
summary=summary.merge(association,on='model',validate='one_to_one')
summary['regime_score_ready_for_blind_test']=False
summary.to_csv(OUT/'ablation_decision_table.csv',index=False)
display(summary)
display(comparison)

,model,fit_train_windows,cluster_count,fit_noise_fraction,largest_assigned_cluster_share,silhouette_assigned_sample,stop_projected_coverage,ridge_train_R2,residual_age_AMI,residual_sector_AMI,age_quintile_AMI,sector_flag_AMI,regime_score_ready_for_blind_test
0,TOSHIBA MG07ACA14TA,13825,2,0.048825,0.963726,0.317063,0.854745,0.851200,0.069104,-0.000116,0.397650,0.000000,False
1,ST12000NM0008,8180,2,0.415770,0.796401,0.608774,0.795139,0.914250,0.133703,0.000000,0.002459,0.918858,False
2,ST4000DM000,8130,2,0.036162,0.989408,0.628017,0.923077,0.964668,0.000314,0.924984,0.080654,0.527171,False
3,ST8000NM0055,6173,2,0.018954,0.971598,0.904535,0.921852,0.889207,0.056316,0.003737,0.037641,0.775733,False
4,HGST HMS5C4040BLE640,5448,2,0.009545,0.984989,0.646590,0.948518,0.875495,0.028458,-0.001034,0.519781,0.000000,False


,model,left_date,right_date,matched_drives,both_assigned_fraction_residual,observed_same_regime,shuffled_same_regime_mean,excess_same_regime_residual,excess_same_regime_original,both_assigned_fraction_original
0,TOSHIBA MG07ACA14TA,2021-08-01,2021-08-08,15354,0.643350,0.986131,0.929671,0.056460,0.541742,0.887130
1,TOSHIBA MG07ACA14TA,2021-08-08,2021-08-15,15351,0.699694,0.990504,0.915137,0.075367,0.545675,0.896228
2,ST12000NM0008,2021-08-01,2021-08-08,9060,0.607506,0.929688,0.903904,0.025784,0.020452,0.941391
3,ST12000NM0008,2021-08-08,2021-08-15,9043,0.618379,0.788984,0.731761,0.057223,0.021069,0.944709
4,ST4000DM000,2021-08-01,2021-08-08,9020,0.853326,0.999870,0.976087,0.023783,0.240073,0.866630
5,ST4000DM000,2021-08-08,2021-08-15,9001,0.885679,0.999749,0.977532,0.022217,0.252008,0.870014
6,ST8000NM0055,2021-08-01,2021-08-08,6810,0.920117,1.000000,0.939922,0.060078,0.197577,0.912188
7,ST8000NM0055,2021-08-08,2021-08-15,6831,0.916118,1.000000,0.940031,0.059969,0.201118,0.913629
8,HGST HMS5C4040BLE640,2021-08-01,2021-08-08,6016,0.597739,1.000000,0.966055,0.033945,0.508104,0.897440
9,HGST HMS5C4040BLE640,2021-08-08,2021-08-15,6044,0.628226,1.000000,0.967258,0.032742,0.494784,0.901721


## 4. Freeze the diagnostic result

This notebook does not choose a final detector or open failure labels. Review whether residual clusters retain substantial membership, nontrivial separation, and temporal persistence, while age/sector associations fall. A follow-up would validate any candidate regime-distance/transition signal with unlabeled synthetic perturbations, frozen calibration on separate drives, and then matched Q3 evaluation before Q4 is opened.

In [5]:
assert sha256(embedding_path)==SETTINGS['embeddings_sha256']
assert sha256(Q3_ZIP)==SETTINGS['q3_zip_sha256']
record={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,
        'settings_sha256':sha256(settings_path),
        'assignments_sha256':sha256(OUT/'residual_regime_assignments.csv'),
        'summary_sha256':sha256(OUT/'residual_cluster_summary.csv'),
        'confounds_sha256':sha256(OUT/'residual_vs_original_confounds.csv'),
        'persistence_sha256':sha256(OUT/'residual_vs_original_persistence.csv'),
        'failure_labels_used':False,'evaluation_drives_used':False,'future_test_opened':False,
        'sealed_June_test_opened':False,'regime_model_approved':False,
        'limitations':['linear residualization removes information as well as nuisance',
                       'three temporal snapshots cannot establish drift or transition warning value',
                       'centroid-radius projection is approximate','no post-ablation resampling stability yet',
                       'no calibration/evaluation drive scoring or blind-test result']}
(OUT/'nuisance_regime_ablation_complete.json').write_text(json.dumps(record,indent=2))
print('NOTEBOOK 22 COMPLETE — nuisance-controlled regime feasibility saved.')
print('No failure labels, evaluation drives, Q4, or sealed June opened. Results:',OUT)

NOTEBOOK 22 COMPLETE — nuisance-controlled regime feasibility saved.
No failure labels, evaluation drives, Q4, or sealed June opened. Results: D:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai\results\single_model_q1q2_2021_v1\q3_nuisance_controlled_regime_ablation_v1
